# 台風被害データ（防災科研 TYDB）取得 → push（Colab版）

[防災科研 台風データベース（TYDB）](https://tydb.bosai.go.jp/TYDB/) の
台風ごとのページ（例: https://tydb.bosai.go.jp/TYDB/HTML/5615.html ）から、
「気象の状況」の要約文と、「被害の状況」の都道府県別被害統計表（死者・
行方不明者、負傷者、全壊・半壊・一部破損、床上・床下浸水、非住家、
情報元）を抽出し、`data/tydb_damage/<台風コード>.json` として保存 →
GitHubへpushします。

既定では全台風（`data/index.json` の全件）を対象に試します。TYDBに
ページが無い台風（404）は `{"notFound": true}` として記録され、次回
以降スキップされます -- `landfallJP=true`（上陸・通過）や
`damaging=true`（大きな被害を出した台風、既存291件）のフラグが立って
いない台風でも、TYDB自体にはページと被害データがあることがあるため
（例: 5111/MARGE, 1951年）、全件を試すのが確実です。

## 使い方
①→②→③→④の順に上から実行してください。

## ① GitHubへのPersonal Access Token（PAT）を用意

すでに持っていれば②に進んでOKです。まだの場合:

1. GitHubの https://github.com/settings/tokens?type=beta を開く
2. **Generate new token** → このリポジトリ（`awg-yk/typhoon-wind-rainfall`）
   に対して **Contents: Read and write** 権限を付与
3. 発行されたトークン（`github_pat_...` から始まる文字列）をコピー

次のセルを実行すると入力欄が出るので、そこに貼り付けてください
（画面には表示されず、Colab上にも保存されません）。

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass('GitHubのPersonal Access Tokenを貼り付けてEnter: ')

## ② リポジトリを取得

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/remaining-tasks-gzid2a'
REMOTE_URL = f'https://{GITHUB_TOKEN}@github.com/awg-yk/typhoon-wind-rainfall.git'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REMOTE_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!cd {REPO_DIR} && git config user.email "colab@example.com"
!cd {REPO_DIR} && git config user.name "Colab"

## ③ 取得実行

デフォルトは `--all`（`data/index.json` の全台風、約1954件）です。
TYDBにページが無い台風は404が返るだけで `{"notFound": true}` として
記録され、次回以降スキップされるので、全件を試すのは安全です（landfallJP
や damaging フラグが立っていない台風でも、TYDB側にはページがあり被害
データが取れることがあるため -- 例: 5111/MARGE, 1951年）。

既に `data/tydb_damage/<コード>.json` があるものは自動スキップします。
1件あたり0.5秒待機するので全体で数十分かかることがあります。

途中で失敗したものがあっても、このセルをもう一度実行すれば失敗分だけ
再試行されます（成功済みはスキップ）。

上陸・通過や大きな被害の台風（291件）だけに絞りたい場合は、下のセルの
`--all` を `--landfall-only` に書き換えてください。

In [ ]:
!pip install -q requests beautifulsoup4
!cd {REPO_DIR} && python3 scripts/fetch_tydb_damage.py --all

import pathlib
out_dir = pathlib.Path(REPO_DIR) / 'data' / 'tydb_damage'
files = list(out_dir.glob('*.json'))
total_kb = sum(f.stat().st_size for f in files) / 1024
print(f'{len(files)} JSON files, {total_kb:.0f} KB total')

## ④ GitHubへコミット & push

`data/tydb_damage/` 配下のJSONだけをコミットします
（`git status` の出力で他のファイルが混ざっていないか一応確認してください）。

In [ ]:
!cd {REPO_DIR} && git add data/tydb_damage/
!cd {REPO_DIR} && git status --short
!cd {REPO_DIR} && git commit -m "Add TYDB per-storm damage data (data/tydb_damage/*.json)"
!cd {REPO_DIR} && git push origin {BRANCH}

## 完了後

- GitHub上の該当ブランチに `data/tydb_damage/*.json` が反映されていれば
  成功です。まだ届いていない分があれば、③→④をもう一度実行すれば続きが
  処理されます。
- このデータは既存の `data/damage_storm_codes.json`（デジタル台風 災害DB
  の死者数ランキング、上位204台風のみ）よりも対象が広く（291台風分）、
  都道府県別の内訳も持っています。次のセッションでこのデータを使って
  被害状況の表示・被害データ入力用シートを更新する想定です。
- このノートブックのセッションを閉じれば、貼り付けたトークンはColab上から
  消えます。念のため、使い終わったトークンはGitHubの設定画面から失効させて
  おくとより安全です。